In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("notebook")

# Choices13k

The following notebook is intended to give a cursory overview of how to interpret and work with the data in choices13k. 

## Data Loading

Load the selection frequencies for the 13,006 problems in choices13k from the file `c13k_selections.csv`. The dataset contains the following columns:<br><br>


<center>

|   Column  |    Data Type    | Description                                                                                                                                                                                                                      |
|:---------:|:---------------:|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
|  Problem  |     Integer     | A unique internal problem ID                                                                                                                                                                                                    |
|  Feedback |     Boolean     | Whether the participants were given feedback about the reward they received and missed out on after making their<br>selection.                                                                                                       |
|     n     |     Integer     | The number of subjects run on the current problem                                                                                                                                                                                |
|   Block   | {1, 2, 3, 4, 5} | The "block ID" for the current problem. For problems with no feedback, Block is always 1. Otherwise, Block ID was<br>sampled uniformly at random from {2, 3, 4, 5}.                                                                 |
|   bRate   |      Float      | The frequency with which subjects selected Gamble B on the current problem.                                       |
|     Ha    |     Integer     | The first outcome of gamble A
                              |
|    pHa    |      Float      | The probability of Ha in gamble A                                                                                   |
|     La    |     Integer     | The second outcome of gamble A (occurs with probability 1-pHa).                                                     |
|     Hb    |     Integer     | The expected value of the lottery in gamble B                                                                                                                                                                                         |
|    pHb    |      Float      | The probability of outcome Lb in gamble B                                                                                                                                                                                             |
|     Lb    |      Float      | The non-lottery outcome for gamble B (occurs with probability pHb)                                                                                                                                                                   |
| LotShapeB |   {0, 1, 2, 3}  | The shape of the lottery distribution for gamble B. A value of 1 indicates the distribution is symmetric around its mean,<br>2 indicates right-skewed, 3 indicates left-skewed, and 0 indicates undefined (i.e., if LotNumB = 1).    |
|  LotNumB  |     Integer     | The number of possible outcomes in the gamble B lottery                                                                                                                                                                               |
|    Amb    |     Boolean     | Whether the decision maker was able to see the probabilities of the outcomes in Gamble B. 1 indicates the participant<br>received no information concerning the probabilities, and 0 implies complete information and no ambiguity. |
|    Corr   |    {-1, 0, 1}   | Whether there is a correlation (negative, zero, or positive) between the payoffs of the two gambles.                                                                                                                            |
|   bRate   |    Float        | The ratio of gamble B selections to total selections for MTurk subjects.                                                                                                                             |
| bRate_std |    Float        | The standard deviation of the ratio of gamble B to total selections for MTurk subjects.                                                                                                                                    |
    
</center>

In [ ]:
c13k_fp = "./c13k_selections.csv"
c13k = pd.read_csv(c13k_fp)

with pd.option_context('display.max_columns', None):
    display(c13k)

Next, load outcomes and their associated probabilities for each problem in c13k from the file `c13k_problems.json`. This data can be joined straightforwardly against the c13k dataframe, as we demonstrate below. 

In `c13k_problems.json`, entries for each gamble are presented as a list of lists. Each sublist is of length 2, where the first entry is an outcome probability, and the second entry is the payout associated with that outcome. 

In [ ]:
c13k_problems = pd.read_json("c13k_problems.json", orient='index')
c13k_problems

In [ ]:
# join the gamble payout probability information against the entries in c13k 
c13k_w_gambles = c13k.join(c13k_problems, how="left")

with pd.option_context('display.max_columns', None):
    display(c13k_w_gambles)

With the combined gamble information and bRate data, we can write a function to provide a more human-readable version of the problem associated with a given c13k entry:

In [ ]:
def print_problem(problem_df, problem_index):
    """
    Print a slightly more readable representation of the gamble information 
    for the gamble associated with index `problem_index` in `problem_df`.
    """
    entry = problem_df.loc[problem_index]
    gA, gB = entry.A, entry.B
    gambleA = pd.DataFrame(gA, columns=["Probability", "Payout"]).sort_values("Probability", ascending=False)
    gambleB = pd.DataFrame(gB, columns=["Probability", "Payout"]).sort_values("Probability", ascending=False)
    linesA = gambleA[["Payout", "Probability"]].to_string().split("\n")
    linesB = gambleB[["Payout", "Probability"]].to_string().split("\n")
    
    cols = ["Problem", "Feedback", "n", "Block", "bRate", "bRate_std"]
    print("{:^55}".format(f"Problem {entry.Problem}, Feedback = {entry.Feedback}"))
    print("{:^55}".format(f"n = {entry.n}, bRate = {entry.bRate:.4f}, std: {entry.bRate_std:.4f}"))
    print(f"\n{'Gamble A':^25} {'Gamble B':>20}")
    for i in range(max(len(linesA), len(linesB))):
        a_str = "" if i >= len(linesA) else linesA[i]
        b_str = "" if i >= len(linesB) else linesB[i]
        print(f"{a_str:<25} {b_str:>25}")

# print a human-readable version of the gamble information for index 0 in c13k_selections.csv.
# output suggests that human MTurk participants (n = 15) selected Gamble B approximately 63% 
# of the time on this problem. 
print_problem(c13k_w_gambles, 0)

## Descriptive Stats

### Feedback vs. no feedback problems

In [ ]:
# inspect the ratio of feedback : no feedback problems in the dataset. as expected,
# no-feedback problems make up approximately 1/5 of the dataset, matching the base
# rates from CPC2015 and 2018.
all_problems = set(c13k.Problem.unique())
feedback_problems = set(c13k[c13k.Feedback == True].Problem.unique())
no_feedback_problems = set(c13k[c13k.Feedback == False].Problem.unique())

feedback_and_no_feedback_problems = feedback_problems.intersection(no_feedback_problems)
feedback_only_problems = (all_problems - feedback_and_no_feedback_problems).intersection(feedback_problems)
no_feedback_only_problems = (all_problems - feedback_and_no_feedback_problems).intersection(no_feedback_problems)

N = len(all_problems)
N_fbk = len(feedback_only_problems)
N_no_fbk = len(no_feedback_only_problems)
N_both = len(feedback_and_no_feedback_problems)

print(f"Total n. problems: {N}")
print(f"N. probs w. ONLY feedback condition: {N_fbk} ({(N_fbk / N) * 100:.2f}%)")
print(f"N. probs w. ONLY no feedback condition: {N_no_fbk} ({(N_no_fbk / N) * 100:.2f}%)")
print(f"N. probs w. BOTH feedback & no feedback conditions: {N_both} ({(N_both / N) * 100:.2f}%)")

### Participants per problem

In [ ]:
# count the number of participants per problem. problems have on average 16 participants, 
# but there is a small subset with almost double this number due to Mechanical Turk errors.
N_ps = c13k.n.mean()
N_ps_feedback = c13k[c13k.Feedback == True].n.mean()
N_ps_no_feedback = c13k[c13k.Feedback == False].n.mean()

N_ps_just_feedback = c13k[c13k.Problem.isin(feedback_only_problems)].n.mean()
N_ps_just_no_feedback = c13k[c13k.Problem.isin(no_feedback_only_problems)].n.mean()
N_ps_just_both = c13k[c13k.Problem.isin(feedback_and_no_feedback_problems)].n.mean()

print(f"Avg. n. participants/problem: {N_ps:.2f}"),
print(f"Avg. n. participants/problem (all feedback problems): {N_ps_feedback:.2f}")
print(f"Avg. n. participants/problem (all no feedback problems): {N_ps_no_feedback:.2f}")
print(f"Avg. n. participants/problem (*only* feedback problems): {N_ps_just_feedback:.2f}")
print(f"Avg. n. participants/problem (*only* no feedback problems): {N_ps_just_no_feedback:.2f}")
print(f"Avg. n. participants/problem (*BOTH* feedback & no feedback problems): {N_ps_just_both:.2f}")
print("")
print(f"Fewest number of participants/problem: {c13k.n.min()}")
print(f"Largest number of participants/problem: {c13k.n.max()}")

In [ ]:
# plot the distribution of subjects per problem, organized by feedback 
# vs. no-feedback conditions. here it is easier to see the subset of 
# problems with a significantly higher participant count.
g = sns.displot(data=c13k, x="n", col="Feedback", kind="kde")
_ = g.fig.subplots_adjust(top=0.8)
_ = g.fig.suptitle('Participants per problem')

### bRate distribution

In [ ]:
# display entries with the largest variability in B-rates.
with pd.option_context('display.max_columns', None):
    print("Entries with largest bRate standard deviation:")
    display(c13k_w_gambles.sort_values("bRate_std", ascending=False).head(10))

In [ ]:
# also inspect problems that had the smallest amount of B-rate variability.
with pd.option_context('display.max_columns', None):
    print("Entries with smallest bRate standard deviation:")
    display(c13k_w_gambles.sort_values("bRate_std", ascending=True).head(10))

# you can use `print_problem` function defined above to inspect the specific gambles 
# print_problem(c13k_w_gambles, index=13814)

In [ ]:
# count the number of problems that all participants agreed on
zero_brate_std = c13k[c13k.bRate_std == 0].shape[0]
zero_brate_std_fb = c13k[(c13k.bRate_std == 0) & (c13k.Feedback == True)].shape[0]
zero_brate_std_no_fb = c13k[(c13k.bRate_std == 0) & (c13k.Feedback == False)].shape[0]
    
print(f"Number of entries w. bRate standard deviation of 0: {zero_brate_std}")
print(f"Number of entries w. bRate standard deviation of 0(feedback): {zero_brate_std_fb}")
print(f"Number of entries w. bRate standard deviation of 0 (no feedback): {zero_brate_std_no_fb}")

In [ ]:
# look at the overall distribution of B-rates, stratified by Feedback
g = sns.displot(data=c13k, x="bRate", col="Feedback", kind="kde")
_ = g.fig.subplots_adjust(top=0.8)
_ = g.fig.suptitle('Problem bRates')